In [ ]:
import pandas as pd
import requests
import time
import concurrent.futures
import json
import os
import re
from tqdm import tqdm

class GitHubDataCollector:
    def __init__(self, token, cache_file='github_cache.json'):
        self.token = token
        self.headers = {
            'Authorization': f'Bearer {token}',
            'Accept': 'application/vnd.github.v3+json'
        }
        self.graphql_headers = {
            'Authorization': f'Bearer {token}',
            'Content-Type': 'application/json'
        }
        self.cache_file = cache_file
        self.cache = self._load_cache()
        self.rate_limit_remaining = 5000
        self.rate_limit_reset = 0
    
    def _load_cache(self):
        if os.path.exists(self.cache_file):
            with open(self.cache_file, 'r') as f:
                return json.load(f)
        return {}
    
    def _save_cache(self):
        with open(self.cache_file, 'w') as f:
            json.dump(self.cache, f)
    
    def _check_rate_limit(self):
        """Check and wait if we're approaching rate limit"""
        if self.rate_limit_remaining < 10:
            wait_time = max(0, self.rate_limit_reset - time.time()) + 5
            print(f"Rate limit almost reached. Waiting {wait_time:.0f} seconds...")
            time.sleep(wait_time)
    
    def _update_rate_limit(self, response):
        """Update rate limit info from response headers"""
        if 'X-RateLimit-Remaining' in response.headers:
            self.rate_limit_remaining = int(response.headers['X-RateLimit-Remaining'])
            self.rate_limit_reset = int(response.headers['X-RateLimit-Reset'])
    
    def get_contributors_count(self, repo):
        """Get contributor count using REST API"""
        owner, name = repo.split('/')
        url = f"https://api.github.com/repos/{owner}/{name}/contributors"
        
        # Use pagination to get total count
        response = requests.get(url, headers=self.headers, params={"per_page": 1})
        self._update_rate_limit(response)
        
        if response.status_code != 200:
            print(f"Error fetching contributors for {repo}: {response.status_code}")
            return 0
        
        # Check if we have Link header for pagination
        if 'Link' in response.headers:
            link_header = response.headers['Link']
            last_page_match = re.search(r'page=(\d+)>; rel="last"', link_header)
            if last_page_match:
                return int(last_page_match.group(1))
        
        # If no pagination or fewer contributors than per_page
        return len(response.json())
    
    def get_repo_info_graphql(self, repo):
        """Get repository info using GraphQL API (much more efficient)"""
        if repo in self.cache:
            return self.cache[repo]
        
        owner, name = repo.split('/')
        
        # Revised GraphQL query that doesn't request collaborators information
        query = """
        query($owner: String!, $name: String!) {
          repository(owner: $owner, name: $name) {
            createdAt
            diskUsage
            primaryLanguage { name }
            languages(first: 10) { totalCount edges { size node { name } } }
            openIssues: issues(states: OPEN) { totalCount }
            closedIssues: issues(states: CLOSED) { totalCount }
            openPullRequests: pullRequests(states: OPEN) { totalCount }
            closedPullRequests: pullRequests(states: CLOSED) { totalCount }
            releases { totalCount }
            defaultBranchRef { target { ... on Commit { history { totalCount } } } }
            forkCount
            stargazerCount
            watchers { totalCount }
          }
        }
        """
        
        variables = {"owner": owner, "name": name}
        data = {"query": query, "variables": variables}
        
        self._check_rate_limit()
        response = requests.post('https://api.github.com/graphql', 
                                json=data, 
                                headers=self.graphql_headers)
        
        if response.status_code != 200:
            print(f"Error fetching {repo}: {response.status_code} {response.text}")
            return {}
        
        result = response.json()
        
        if 'errors' in result:
            print(f"GraphQL errors for {repo}: {result['errors']}")
            # We'll continue anyway and just use what we can get
        
        if 'data' not in result or 'repository' not in result['data'] or result['data']['repository'] is None:
            print(f"No repository data returned for {repo}")
            return {}
            
        repo_data = result['data']['repository']
        
        # Extract and format the data
        languages_data = {}
        if repo_data.get('languages') and 'edges' in repo_data['languages']:
            for edge in repo_data['languages']['edges']:
                languages_data[edge['node']['name']] = edge['size']
        
        # Get contributors count using REST API as a fallback
        contributors_count = self.get_contributors_count(repo)
        
        info = {
            'creation_date': repo_data.get('createdAt'),
            'size': repo_data.get('diskUsage'),
            'language': repo_data.get('primaryLanguage', {}).get('name') if repo_data.get('primaryLanguage') else None,
            'LOC': languages_data,
            'contributors': contributors_count,
            'openIssues': repo_data.get('openIssues', {}).get('totalCount', 0) if repo_data.get('openIssues') else 0,
            'closedIssues': repo_data.get('closedIssues', {}).get('totalCount', 0) if repo_data.get('closedIssues') else 0,
            'commits': repo_data.get('defaultBranchRef', {}).get('target', {}).get('history', {}).get('totalCount', 0) 
                      if repo_data.get('defaultBranchRef') and repo_data.get('defaultBranchRef', {}).get('target') else 0,
            'openPRs': repo_data.get('openPullRequests', {}).get('totalCount', 0) if repo_data.get('openPullRequests') else 0,
            'closedPRs': repo_data.get('closedPullRequests', {}).get('totalCount', 0) if repo_data.get('closedPullRequests') else 0,
            'releases': repo_data.get('releases', {}).get('totalCount', 0) if repo_data.get('releases') else 0,
            'forks': repo_data.get('forkCount', 0),
            'stars': repo_data.get('stargazerCount', 0),
            'watchers': repo_data.get('watchers', {}).get('totalCount', 0) if repo_data.get('watchers') else 0
        }
        
        # Cache the results
        self.cache[repo] = info
        self._save_cache()
        
        return info
    
    def process_repositories(self, repos, max_workers=3):
        """Process multiple repositories concurrently"""
        results = {}
        
        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
            # Submit all repositories for processing
            future_to_repo = {
                executor.submit(self.get_repo_info_graphql, repo): repo 
                for repo in repos if repo
            }
            
            # Process as they complete
            for future in tqdm(concurrent.futures.as_completed(future_to_repo), 
                              total=len(future_to_repo),
                              desc="Processing repositories"):
                repo = future_to_repo[future]
                try:
                    results[repo] = future.result()
                except Exception as e:
                    print(f"Error processing {repo}: {str(e)}")
                    results[repo] = {}
        
        return results

# Usage
token = "ghp_xxxx"  # Replace with your token
collector = GitHubDataCollector(token)

# Load your CSV
df = pd.read_csv('top250_projects.csv')

# Extract repo names from links
repos = [link.split('https://github.com/')[-1] if isinstance(link, str) and 'github.com' in link else None 
         for link in df['link']]
repos = [r for r in repos if r]  # Filter out None values

# Process repositories
results = collector.process_repositories(repos, max_workers=3)  # Adjust workers as needed

# Update the dataframe with results
for index, row in df.iterrows():
    if isinstance(row['link'], str) and 'github.com' in row['link']:
        repo = row['link'].split('https://github.com/')[-1]
        if repo in results:
            for key, value in results[repo].items():
                df.at[index, key] = value if not isinstance(value, dict) else json.dumps(value)

# Save the updated dataframe
df.to_csv('top250_projects.csv', index=False)
os.remove('github_cache.json')

In [2]:
import pandas as pd

df = pd.read_csv('top250_projects.csv')

language_category = {
    "TypeScript": "GPL",
    "Python": "GPL",
    "Go": "GPL",
    "PHP": "GPL",
    "JavaScript": "GPL",
    "HTML": "DSL",
    "Shell": "DSL",
    "XSLT": "DSL",
    "Groovy": "GPL",
    "C++": "GPL",
    "Jupyter Notebook": "DSL",
    "Scala": "GPL",
    "C#": "GPL",
    "Java": "GPL",
    "Swift": "GPL",
    "PowerShell": "DSL",
    "Rust": "GPL",
    "Batchfile": "DSL",
    "YAML": "DSL",
    "Vue": "DSL",
    "Ruby": "GPL",
    "Vim Script": "DSL",
    "Vim script": "DSL",
    "Dockerfile": "DSL",
    "Jinja": "DSL",
    "Scheme": "GPL",
    "Kotlin": "GPL",
    "Starlark": "DSL",
    "C": "GPL",
    "Makefile": "DSL",
    "Nix": "DSL",
    "CSS": "DSL",
    "Lua": "GPL",
    "MDX": "DSL",
    "Verilog": "DSL",
    "CMake": "DSL",
    "Roff": "DSL",
    "Jsonnet": "DSL"
}

df['PL_category'] = df['language'].map(language_category)

language_typing = {
    "TypeScript": "both",
    "Python": "dynamic",
    "Go": "static",
    "PHP": "dynamic",
    "JavaScript": "dynamic",
    "Groovy": "dynamic",
    "C++": "static",
    "Scala": "both",
    "C#": "static",
    "Java": "static",
    "Swift": "static",
    "Rust": "static",
    "Ruby": "dynamic",
    "Scheme": "dynamic",
    "Kotlin": "static",
    "C": "static",
    "Lua": "dynamic"
}

df['PL_typting'] = df['language'].map(language_typing)

df.to_csv('top250_projects.csv', index=False)

In [2]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# Read the CSV file
df = pd.read_csv('top250_projects.csv')

# Filter data for GPL and DSL projects
gpl_data = df[df['PL_category'] == 'GPL']
dsl_data = df[df['PL_category'] == 'DSL']

# Function to create language distribution chart
def create_language_chart(data, type):
    # Group by language and type, then count occurrences
    dist = data.groupby(['language', 'type']).size().reset_index(name='count')
    
    # Calculate total count per language to determine sorting order
    language_totals = dist.groupby('language')['count'].sum().reset_index()
    language_totals = language_totals.sort_values('count', ascending=False)
    
    # Create ordered list of languages based on total occurrence
    language_order = language_totals['language'].tolist()
    
    # Create the figure
    fig = go.Figure()
    
    # Define colors for SPDX and CycloneDX
    colors = {'SPDX': '#1f77b4', 'CycloneDX': '#ff7f0e'}
    
    # Get unique types
    types = dist['type'].unique()
    
    # Add bars for each type, using the ordered languages
    for type_name in types:
        type_data = dist[dist['type'] == type_name]
        
        # Create arrays for x and y values in the correct order
        x_values = []
        y_values = []
        text_values = []
        
        for lang in language_order:
            lang_data = type_data[type_data['language'] == lang]
            if not lang_data.empty:
                x_values.append(lang)
                y_values.append(lang_data['count'].iloc[0])
                text_values.append(lang_data['count'].iloc[0])
            else:
                x_values.append(lang)
                y_values.append(0)
                text_values.append('')
        
        fig.add_trace(
            go.Bar(
                x=x_values,
                y=y_values,
                name=type_name,
                marker_color=colors.get(type_name, '#1f77b4'),
                text=text_values,
                textposition='outside'
            )
        )
    
    # Update layout
    fig.update_layout(
        # title={
        #     'text': title,
        #     'x': 0.5,
        #     'xanchor': 'center',
        #     'font': {'size': 16}  # UNCOMMENT THIS FOR TITLE FONT
        # },
        xaxis_title=f"{type} Language",
        yaxis_title="Count",
        barmode='group',
        height=500,
        width=800,
        legend=dict(
            orientation="v",
            yanchor="top",
            y=0.95,
            xanchor="right",
            x=0.95,
            bgcolor="rgba(255,255,255,0.8)",
            bordercolor="rgba(0,0,0,0.2)",
            borderwidth=1
        ),
        # Add these new parameters for axis labeling fonts
        xaxis_title_font=dict(size=20),  # X-axis title font size
        yaxis_title_font=dict(size=20),  # Y-axis title font size
        xaxis_tickfont=dict(size=14),    # X-axis labels font size
        yaxis_tickfont=dict(size=14)     # Y-axis labels font size
    )
    
    # Rotate x-axis labels if there are many languages
    if len(language_order) > 5:
        fig.update_xaxes(tickangle=45)
    
    return fig, dist, language_totals

# Create GPL chart
gpl_fig, gpl_dist, gpl_totals = create_language_chart(gpl_data, type='GPL')

# Create DSL chart
dsl_fig, dsl_dist, dsl_totals = create_language_chart(dsl_data, type='DSL')

# Show both figures
gpl_fig.show()
dsl_fig.show()

# Save the figures as pdf files
gpl_fig.write_image("gpl_projects_chart.pdf")
dsl_fig.write_image("dsl_projects_chart.pdf")

In [44]:
def create_language_type_statistics():
    # Filter out rows with missing language data
    df_with_lang = df.dropna(subset=['language'])
    
    # Create statistics by language and type
    lang_type_stats = df_with_lang.groupby(['language', 'type']).size().reset_index(name='count')
    
    # Pivot to get CycloneDX and SPDX as columns
    lang_stats_pivot = lang_type_stats.pivot(index='language', columns='type', values='count').fillna(0)
    
    # Add total column
    lang_stats_pivot['Total'] = lang_stats_pivot.sum(axis=1)
    
    # Sort by total count descending
    lang_stats_pivot = lang_stats_pivot.sort_values('Total', ascending=False).reset_index()
    
    return lang_stats_pivot

# Create the language-type statistics
language_type_stats = create_language_type_statistics()
language_type_stats


type,language,CycloneDX,SPDX,Total
0,Go,80.0,62.0,142.0
1,Python,45.0,34.0,79.0
2,Java,21.0,47.0,68.0
3,Shell,15.0,14.0,29.0
4,TypeScript,14.0,11.0,25.0
5,C#,6.0,14.0,20.0
6,JavaScript,12.0,6.0,18.0
7,HTML,4.0,8.0,12.0
8,Makefile,7.0,3.0,10.0
9,PowerShell,4.0,2.0,6.0


In [34]:
import pandas as pd
import ast

# Read the CSV file
df_cdx = pd.read_csv(r'..\RQ3-issues\CdxIssues.csv')
df_spdx = pd.read_csv(r'..\RQ3-issues\SpdxIssues.csv')

df_cdx = df_cdx[df_cdx['created_at'] >= '2018-01-01']
df_spdx = df_spdx[df_spdx['created_at'] >= '2018-01-01']

df_cdx['Tags'] = df_cdx['Tags'].apply(lambda x: ast.literal_eval(x))
df_spdx['Tags'] = df_spdx['Tags'].apply(lambda x: ast.literal_eval(x))

df_cdx_tag = df_cdx[df_cdx['Tags'].apply(lambda x: len(x) > 0)]
df_spdx_tag = df_spdx[df_spdx['Tags'].apply(lambda x: len(x) > 0)]

# df_cdx_tag = df_cdx['Tags'].apply(lambda x: len(x))
# df_cdx_tag.unique()

# calculate the percentage of total issues with tags
print(f"Total issues: {df_cdx.shape[0]+df_spdx.shape[0]}")
print(f"CDX issues: {df_cdx.shape[0]}")
print(f"SPDX issues: {df_spdx.shape[0]}")
print(f"Total issues with tags: {df_cdx_tag.shape[0]+df_spdx_tag.shape[0]}")
print(f"CDX issues with tags: {df_cdx_tag.shape[0]}")
print(f"SPDX issues with tags: {df_spdx_tag.shape[0]}")

Total issues: 33840
CDX issues: 17093
SPDX issues: 16747
Total issues with tags: 25478
CDX issues with tags: 13732
SPDX issues with tags: 11746
